# 03 — Day-ahead load forecasting: deep models (N-HiTS, N-BEATS, TFT) and final comparison

Same backtest as `02_forecast_baselines.ipynb` (D-1 12:00 origin, steps 13–36, test 2024+), through
NeuralForecast `cross_validation`.

> ⚠️ This notebook uses **torch** and must not import LightGBM (macOS libomp clash). The final
> comparison reads the baselines stored by notebook 02 from `data/interim/forecasts/` (pandas only),
> so run notebook 02 first.

| Model | device | refit | measured speed (M4) | approx. runtime |
|---|---|---|---|---|
| N-HiTS, N-BEATSx, N-BEATS-I | CPU | 90 days | 20–100 steps/s | ~30–45 min |
| TFT (hidden 32, input 168) | MPS | 180 days | ~3.5 steps/s | ~40 min |

The TFT gives native quantiles (MQLoss); the other models get causal rolling conformal intervals.

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import pandas as pd

from ons_energy.forecast.backtest import FORECAST_DIR, last_full_day, mase_scales, merge_forecasts, run_or_load
from ons_energy.forecast.config import BacktestConfig, ModelSpec
from ons_energy.forecast.evaluate import attach_day_types, dm_table, evaluate_backtest
from ons_energy.tracking import log_backtest
from ons_energy.viz import forecast_plots as F

pd.set_option("display.precision", 3)
RETRAIN = False  # True: refit even if a stored backtest exists
LOG_MLFLOW = True
RUN_INTERPRET = True  # TFT variable selection / attention and N-BEATS decomposition (refits once, ~10 min)
RUN_GLOBAL = False  # N-HiTS / TFT on 4 subsystems with day-before weather forecasts (several hours)

## N-HiTS, N-BEATSx and interpretable N-BEATS (CPU)

In [ ]:
cfg_dl = BacktestConfig(
    name="seco_dl",
    subsystems=["SE"],
    accelerator="cpu",
    refit_every_days=90,
    models=[
        ModelSpec(name="NHITS", kind="nhits"),
        ModelSpec(name="NBEATSx", kind="nbeatsx"),
        ModelSpec(name="NBEATS_I", kind="nbeats_i"),
    ],
)
cv_dl = run_or_load(cfg_dl, retrain=RETRAIN)
tabs_dl = evaluate_backtest(cv_dl, mase_scales(cfg_dl))
if LOG_MLFLOW:
    log_backtest(cfg_dl, tabs_dl, cv_dl)
tabs_dl["overall"].sort_values("MAPE")

In [ ]:
DL = ["NHITS", "NBEATSx", "NBEATS_I"]
fig = F.actual_vs_forecast_daily(cv_dl, DL)

In [ ]:
fig = F.actual_vs_forecast(cv_dl, DL, "2025-02-27", band_model="NHITS", title="SE/CO — Carnival 2025 (N-HiTS conformal intervals shaded)")

## Temporal Fusion Transformer (MPS)

In [ ]:
cfg_tft = BacktestConfig(
    name="seco_tft",
    subsystems=["SE"],
    accelerator="mps",
    refit_every_days=180,
    models=[ModelSpec(name="TFT", kind="tft", params={"hidden_size": 32, "n_head": 4, "input_size": 168, "windows_batch_size": 64, "max_steps": 1000})],
)
cv_tft = run_or_load(cfg_tft, retrain=RETRAIN)
tabs_tft = evaluate_backtest(cv_tft, mase_scales(cfg_tft))
if LOG_MLFLOW:
    log_backtest(cfg_tft, tabs_tft, cv_tft)
tabs_tft["overall"]

In [ ]:
fig = F.actual_vs_forecast_daily(cv_tft, ["TFT"])

In [ ]:
fig = F.actual_vs_forecast(cv_tft, ["TFT"], "2024-03-11", band_model="TFT", title="SE/CO — a normal week (TFT native 80/90% quantiles)")

In [ ]:
fig = F.actual_vs_forecast(cv_tft, ["TFT"], "2026-06-23", days=8, band_model="TFT",
                           title="SE/CO — World Cup 2026: BRA×SCO 24/06 19:00, BRA×JPN 29/06 14:00")

## Interpretability
* **TFT**: variable-selection weights of the future (decoder) inputs, normal vs special days. NeuralForecast
  keeps them only for the last predicted batch, so each origin (all special days + 25 normal days) is
  predicted separately from a model fitted up to the test start.
* **Interpretable N-BEATS**: the forecast split into trend and seasonality stacks.

In [ ]:
if RUN_INTERPRET:
    from ons_energy.forecast.interpret import neural_interpretability

    end = cfg_tft.eval_end(last_full_day())
    interp_tft = neural_interpretability(cfg_tft, end)
    fig = F.tft_vsn(interp_tft["tft_vsn"])

In [ ]:
if RUN_INTERPRET:
    v = interp_tft["tft_vsn_by_day_type"]
    display(v[v.block == "future"].pivot_table(index="variable", columns="day_type", values="weight").round(3))

In [ ]:
if RUN_INTERPRET:
    att = interp_tft["tft_attention"]
    ax = att.pivot_table(index="lag_h", columns="segment", values="weight").plot(
        figsize=(9, 3.4), title="TFT mean attention by hours before the origin")

In [ ]:
if RUN_INTERPRET:
    interp_nb = neural_interpretability(cfg_dl, cfg_dl.eval_end(last_full_day()))
    fig = F.nbeats_decomposition(interp_nb["nbeats_decomposition"])

## Final comparison: all models on the same SE/CO windows
Baselines from notebook 02 (`mvp_seco_baselines`) + `seco_dl` + `seco_tft`, inner-joined on the
same target hours. Reference for Diebold-Mariano: the ONS programmed load.

In [ ]:
names = [n for n in ("mvp_seco_baselines", "seco_dl", "seco_tft") if (FORECAST_DIR / f"{n}.parquet").exists()]
assert "mvp_seco_baselines" in names, "run 02_forecast_baselines.ipynb first"
cv_all = merge_forecasts(names)
tabs_all = evaluate_backtest(cv_all, mase_scales(cfg_dl), reference="ONS")
overall = tabs_all["overall"].sort_values("MAPE")
overall[["model", "n_days", "MAPE", "sMAPE", "RMSE", "MASE", "pinball", "cov80", "cov90"]]

In [ ]:
tabs_all["dm"].sort_values("DM")  # vs ONS programmed load

In [ ]:
dm_table(attach_day_types(cv_all), "SeasonalNaive").sort_values("DM")

In [ ]:
mape = overall.set_index("model")["MAPE"]
best_dl = min([m for m in ("NHITS", "NBEATSx", "NBEATS_I", "TFT") if m in mape.index], key=mape.get)
SHOW = list(dict.fromkeys(["ONS_adj", "LightGBM", best_dl, "TFT"] if "TFT" in mape.index else ["ONS_adj", "LightGBM", best_dl]))
BAND = "TFT" if "TFT-q0.05" in cv_all else best_dl
print("best deep model:", best_dl, "| shown:", SHOW)
fig = F.actual_vs_forecast_daily(cv_all, SHOW)

In [ ]:
fig = F.actual_vs_forecast(cv_all, SHOW, "2025-02-27", band_model=BAND, title=f"SE/CO — Carnival 2025, all models ({BAND} intervals)")

In [ ]:
fig = F.actual_vs_forecast(cv_all, SHOW, "2026-06-23", days=8, band_model=BAND,
                           title="SE/CO — World Cup 2026: BRA×SCO 24/06 19:00, BRA×JPN 29/06 14:00")

In [ ]:
seg = tabs_all["by_day_type"]
display(seg.pivot_table(index="day_type", columns="model", values="MAPE", observed=True).assign(
    n_days=seg.groupby("day_type", observed=True)["n_days"].max()).round(2))
fig = F.mape_by_day_type(seg, SHOW, "SE/CO MAPE by day type — all models")

In [ ]:
fig = F.mape_by_hour(tabs_all["by_step"], SHOW, "SE/CO MAPE by hour of the target day — all models")

## Optional: global deep models (4 subsystems, day-before weather forecasts)

In [ ]:
if RUN_GLOBAL:
    SUBS = ["SE", "S", "NE", "N"]
    globals_ = [
        BacktestConfig(name="global_nhits_none", subsystems=SUBS, accelerator="cpu", refit_every_days=90, models=[ModelSpec(name="NHITS", kind="nhits")]),
        BacktestConfig(name="global_nhits_honest", subsystems=SUBS, weather="honest", accelerator="cpu", refit_every_days=90, models=[ModelSpec(name="NHITS", kind="nhits")]),
        BacktestConfig(name="global_tft_honest", subsystems=SUBS, weather="honest", accelerator="mps", refit_every_days=180,
                       models=[ModelSpec(name="TFT", kind="tft", params={"hidden_size": 32, "n_head": 4, "input_size": 168, "windows_batch_size": 64, "max_steps": 1500})]),
    ]
    rows = {}
    for c in globals_:
        cv_g = run_or_load(c, retrain=RETRAIN)
        t = evaluate_backtest(cv_g, mase_scales(c))
        if LOG_MLFLOW:
            log_backtest(c, t, cv_g)
        rows[c.name] = t["by_subsystem"].pivot_table(index="model", columns="unique_id", values="MAPE")
    display(pd.concat(rows).round(2))
    fig = F.actual_vs_forecast_daily(cv_g, ["TFT"], subsystem="NE")